# 第14章　信用风险与定价模型

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch14_credit_risk.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch14_credit_risk.ipynb)

复现例12.1（Merton）、例12.2（约化模型）、例12.3（隐含违约率曲线）、图14-1，QuantLib 对拍与回收率敏感性。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import credit as cr
from fi import plotting
plotting.use_chinese_style()


## 例12.1　Merton 结构化模型：违约距离与违约概率


In [ ]:
print(f"{'资产V':>6}{'杠杆D/V':>9}{'违约距离':>9}{'违约概率':>9}{'Merton利差':>11}")
for V in (120, 150, 200):
    res = cr.merton_pd(V, 100, 0.25, 0.03, 1)
    sp = cr.merton_credit_spread(V, 100, 0.25, 0.03, 1)
    print(f'{V:>6}{100/V:>9.2f}{res["distance_to_default"]:>9.2f}{res["pd"]*100:>8.2f}%{sp*1e4:>9.0f}bp')


## 图14-1　违约概率与利差随杠杆上升（编程实验 7）


In [ ]:
lev = np.linspace(0.4, 0.92, 40); D = 100.0
V = D / lev
pd = [cr.merton_pd(v, D, 0.25, 0.03, 1)['pd']*100 for v in V]
spread = [cr.merton_credit_spread(v, D, 0.25, 0.03, 1)*1e4 for v in V]
fig, ax1 = plotting.new_axes()
ax1.plot(lev, pd, 'C0', label='违约概率 PD')
ax1.set_xlabel('杠杆 D/V'); ax1.set_ylabel('违约概率 PD (%)', color='C0')
ax2 = ax1.twinx(); ax2.plot(lev, spread, 'C3--')
ax2.set_ylabel('信用利差 (bp)', color='C3')
ax1.set_title('图14-1　Merton：违约概率与利差随杠杆上升')
fig.tight_layout()


## 例12.2　约化模型：利差 -> 违约强度 -> 违约概率


In [ ]:
lam = cr.hazard_from_spread(0.02, recovery=0.4)
print(f'利差200bp, 回收率40% -> 违约强度 λ = {lam*100:.3f}%')
for t in (1, 3, 5):
    print(f'  {t}年: 生存概率={cr.survival_probability(lam,t)*100:.2f}%  累计违约={cr.default_probability(lam,t)*100:.2f}%')


## 例12.3　由利差曲线 bootstrap 隐含违约率曲线（编程实验 8）

并比较回收率 30%/40%/50% 的影响。


In [ ]:
tenors = [1, 2, 3, 5, 7, 10]
spreads = [0.005, 0.008, 0.011, 0.015, 0.018, 0.022]
fig, ax = plotting.new_axes()
for R in (0.30, 0.40, 0.50):
    t, pd = cr.implied_default_curve(tenors, spreads, recovery=R)
    ax.plot(t, pd*100, marker='o', label=f'回收率 {R:.0%}')
    if R == 0.40:
        for ti, pdi in zip(t, pd):
            print(f'  {ti:.0f}年: 累计违约概率(R=40%)={pdi*100:.2f}%')
ax.set_xlabel('期限（年）'); ax.set_ylabel('累计违约概率 (%)')
ax.set_title('隐含违约率曲线（回收率敏感性）'); ax.legend()
fig.tight_layout()


## 12.7　QuantLib 违约期限结构对拍


In [ ]:
import QuantLib as ql
today = ql.Date(15, 6, 2026); ql.Settings.instance().evaluationDate = today
dpts = ql.FlatHazardRate(today, ql.QuoteHandle(ql.SimpleQuote(lam)), ql.Actual365Fixed())
print(f"{'期限':>4}{'fi 生存':>10}{'QL 生存':>10}")
for y in (1, 3, 5):
    d = today + ql.Period(y, ql.Years)
    print(f'{y:>4}{cr.survival_probability(lam,y)*100:>9.2f}%{dpts.survivalProbability(d)*100:>9.2f}%')


---

> 小结：结构化模型由资产/负债推违约距离与 PD（杠杆越高越危险），约化模型由利差反推违约强度；
> 利差反推的是风险中性违约率（高于真实），`fi.credit` 与 QuantLib 违约期限结构一致。
